In [1]:
import mlflow
mlflow.set_tracking_uri("sqlite:///C:/Dev/taxi-fare-predictor/mlflow.db")
mlflow.set_experiment("taxi-fare-prediction")

<Experiment: artifact_location='file:c:/Dev/taxi-fare-predictor/notebooks/mlruns/1', creation_time=1786302150001, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786302150001, lifecycle_stage='active', name='taxi-fare-prediction', tags={}, trace_location=None, workspace='default'>

In [2]:
import sys
sys.path.append('../src')

from train import build_dataset, time_sorted_split, train_linear_regession, train_RandomForest, train_XGBoost
from train import TARGET, NUMERICAL_FEATURES, CATEGORICAL_FEATURES, TIME_SORT_INDEX

In [3]:
df = build_dataset()

X_train, y_train, X_test, y_test = time_sorted_split(
    df,
    num_features=NUMERICAL_FEATURES,
    cat_features=CATEGORICAL_FEATURES,
    target=TARGET,
    sort_index=TIME_SORT_INDEX,
    test_proportion=0.2
)

In [4]:
lr_model, lr_rmse, lr_mae, lr_r2 = train_linear_regession(X_train, y_train, X_test, y_test)

2026/08/09 21:53:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Train RMSE: 4.150 | Test RMSE: 4.874 | Gap: 0.725
Train R2: 0.938 | Test R2: 0.914


In [5]:
rf_model, rf_rmse, rf_mae, rf_r2 = train_RandomForest(X_train, y_train, X_test, y_test)

2026/08/09 21:56:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Train RMSE: 1.752 | Test RMSE: 2.814 | Gap: 1.062
Train R2: 0.989 | Test R2: 0.971


In [6]:
xgb_model, xgb_rmse, xgb_mae, xgb_r2 = train_XGBoost(X_train, y_train, X_test, y_test)

2026/08/09 21:57:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Train RMSE: 1.921 | Test RMSE: 2.227 | Gap: 0.305
Train R2: 0.987 | Test R2: 0.982


In [7]:
experiment = mlflow.get_experiment_by_name("taxi-fare-prediction")
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
runs[['tags.mlflow.runName', 'metrics.rmse', 'metrics.mae', 'metrics.r2']].sort_values('metrics.rmse')

,tags.mlflow.runName,metrics.rmse,metrics.mae,metrics.r2
5,RandomForest,2.357648,0.839201,0.979828
4,LinearRegression,4.874208,2.730622,0.913783
6,LinearRegression,4.874208,2.730622,0.913783
7,linear_baseline,4.874208,2.730622,0.913783
9,quick_notebook_test,7.279784,NaN,NaN
12,quick_notebook_test,52.995256,NaN,NaN
0,XGBoost,NaN,NaN,NaN
1,RandomForest,NaN,NaN,NaN
2,LinearRegression,NaN,NaN,NaN
3,RandomForest,NaN,NaN,NaN


# Model Registry

In [8]:
from mlflow import MlflowClient

client = MlflowClient()

experiment = mlflow.get_experiment_by_name("taxi-fare-prediction")
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

lr_run_id = runs[runs['tags.mlflow.runName'] == 'LinearRegression'].iloc[0]['run_id']
rf_run_id = runs[runs['tags.mlflow.runName'] == 'RandomForest'].iloc[0]['run_id']
xgb_run_id = runs[runs['tags.mlflow.runName'] == 'XGBoost'].iloc[0]['run_id']

mlflow.register_model(f"runs:/{lr_run_id}/model", "taxi-fare-linear")
mlflow.register_model(f"runs:/{rf_run_id}/model", "taxi-fare-randomforest")
mlflow.register_model(f"runs:/{xgb_run_id}/model", "taxi-fare-xgboost")

Successfully registered model 'taxi-fare-linear'.
2026/08/09 22:03:25 WARNING mlflow.tracking._model_registry.fluent: Run with id e2f075febe094324afc6377119e6255d has no artifacts at artifact path 'model', registering model based on models:/m-e6be1c43b30a4f86a09238aaff678bba instead
Created version '1' of model 'taxi-fare-linear'.
Successfully registered model 'taxi-fare-randomforest'.
2026/08/09 22:03:25 WARNING mlflow.tracking._model_registry.fluent: Run with id 28ef315cea6a4dd9845ebfc4b198893b has no artifacts at artifact path 'model', registering model based on models:/m-ffd73cde215b4a668c6b5e3c3ffb21d0 instead
Created version '1' of model 'taxi-fare-randomforest'.
Successfully registered model 'taxi-fare-xgboost'.
2026/08/09 22:03:25 WARNING mlflow.tracking._model_registry.fluent: Run with id d1def0aad03c4c919c926f526862aaa4 has no artifacts at artifact path 'model', registering model based on models:/m-e855fa8c7b3b4808b4d8a3843fe147d3 instead
Created version '1' of model 'taxi-fa

<ModelVersion: aliases=[], creation_timestamp=1786309405907, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1786309405907, metrics=None, model_id=None, name='taxi-fare-xgboost', params=None, run_id='d1def0aad03c4c919c926f526862aaa4', run_link=None, source='models:/m-e855fa8c7b3b4808b4d8a3843fe147d3', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>

In [9]:
client.update_model_version(name="taxi-fare-linear", version=1, description="Untuned baseline. Test RMSE 4.874, R2 0.914.")
client.update_model_version(name="taxi-fare-randomforest", version=1, description="Untuned baseline, max_depth=20. Test RMSE 2.814, R2 0.971. Larger train/test gap - some overfitting.")
client.update_model_version(name="taxi-fare-xgboost", version=1, description="Untuned baseline. Test RMSE 2.227, R2 0.982. Smallest train/test gap - best generalization.")

client.set_model_version_tag("taxi-fare-linear", version=1, key="tuning_status", value="untuned_baseline")
client.set_model_version_tag("taxi-fare-randomforest", version=1, key="tuning_status", value="untuned_baseline")
client.set_model_version_tag("taxi-fare-xgboost", version=1, key="tuning_status", value="untuned_baseline")